# Notebook da atividade prática - Programação assistida por inteligência artificial

- **Aluno:** Pedro Henrique de Oliveira
- **Curso:** Ciência da Computação - UDF
- **Disciplina:** Tendências em Ciência da Computação
- **Professora:** Kadidja Valéria
- **Unidade:** 2
- **Data:** 17/09/2026
- **Miniprojeto escolhido:** Calculadora Simples - nível básico
- **Ferramentas utilizadas nesta entrega:** Python, terminal e IA como assistente.


## Como usar este notebook

Este arquivo reúne o relatório e os **códigos completos e comentados**, com exemplos executáveis para comparar a base com a versão final. Ele funciona de forma independente.

1. Abra o arquivo no Jupyter, no suporte a notebooks da sua IDE ou envie o `.ipynb` ao Google Colab.
2. Selecione um ambiente Python 3.10 ou superior e execute as células de cima para baixo.
3. As demonstrações usam entradas programadas, e os testes rodam sem pedir digitação. Assim, **Executar tudo** consegue chegar ao final.
4. Para usar a calculadora pelo teclado, siga as instruções da seção **Uso interativo opcional**.

**Adaptação para notebook:** a função da base foi nomeada `calcular_base`, para não sobrescrever `calcular` da versão final. Os blocos `if __name__ == "__main__"` foram substituídos por células de execução.


## 1. Objetivo e escolha do projeto

A proposta da atividade é usar a IA como apoio durante a programação, construir uma base e depois avaliar e melhorar o código. O trabalho segue o ciclo apresentado no material: contextualizar, avaliar criticamente, refatorar e decidir o que integrar.

Escolhi a Calculadora Simples porque ela permite trabalhar diretamente com o tratamento de exceções, validação de tipos de dados e modularização de operações matemáticas. O objetivo foi entregar uma calculadora de terminal robusta, que não "quebrasse" com erros comuns do usuário.

A atividade foi desenvolvida com assistência de IA, incluindo implementação, revisão e execução dos testes.


## 2. Regras definidas

1. O programa recebe a operação desejada e, em seguida, os números necessários.
2. Suporta adição, subtração, multiplicação e divisão, além de exponenciação e raiz quadrada (sugeridas pela IA).
3. A divisão por zero não deve interromper o programa abruptamente com um erro do interpretador.
4. Entradas de texto quando um número é esperado não devem causar o encerramento do programa (ValueError).
5. O laço de repetição continua até que o usuário escolha a opção de sair.


In [ ]:
# Bibliotecas para demonstrar o uso e executar os testes.
import io
import sys
import unittest
from contextlib import redirect_stdout
from unittest.mock import patch
import math

print("Python utilizado:", sys.version.split()[0])


## 3. Versão inicial

A célula abaixo contém a função da versão base. Ela concentra a entrada de dados, a lógica das operações e as exibições no mesmo bloco `while`.

Essa versão permite realizar cálculos, mas apresenta limitações críticas:
- Se o usuário digitar uma letra ao invés de um número, o programa sofre um *crash* (`ValueError`).
- Se o usuário tentar dividir por zero, o programa sofre um *crash* (`ZeroDivisionError`).
- A mistura entre interface (prints/inputs) e regras matemáticas dificulta testar as operações isoladamente.

### Evidência do problema

Na simulação de execução da base com as entradas `/`, `10` e `0`, o programa é abortado instantaneamente pelo sistema, prejudicando a experiência. A lógica inicial está preservada em `calcular_base` para comparação.


In [ ]:
# VERSÃO BASE: Procedural e frágil contra erros.
def calcular_base():
    while True:
        print("\n=== Calculadora Base ===")
        print("Opções: +, -, *, / ou '0' para sair")
        op = input("Operação: ")
        
        if op == '0':
            print("Saindo...")
            break
            
        if op in ['+', '-', '*', '/']:
            # Ponto de falha 1: ValueError se digitar texto
            n1 = float(input("Primeiro número: "))
            n2 = float(input("Segundo número: "))
            
            if op == '+':
                print("Resultado:", n1 + n2)
            elif op == '-':
                print("Resultado:", n1 - n2)
            elif op == '*':
                print("Resultado:", n1 * n2)
            elif op == '/':
                # Ponto de falha 2: ZeroDivisionError se n2 for 0
                print("Resultado:", n1 / n2)
        else:
            print("Opção inválida.")


### Demonstração da base: quebra por divisão por zero

O auxiliar abaixo simula a digitação. Observe que a tentativa de dividir por zero não é tratada pela calculadora, resultando em uma exceção que força a parada da aplicação.


In [ ]:
def simular_base(entradas):
    entradas_iter = iter(entradas)
    saida = io.StringIO()

    def entrada_simulada(mensagem):
        val = next(entradas_iter)
        print(mensagem + val)
        return val

    with patch("builtins.input", side_effect=entrada_simulada):
        with redirect_stdout(saida):
            try:
                calcular_base()
            except Exception as e:
                print(f"\n[CRASH DO SISTEMA] {type(e).__name__}: {e}")
                
    return saida.getvalue()

saida_base = simular_base(['/', '10', '0'])
print(saida_base)
assert "ZeroDivisionError" in saida_base


## 4. Roteiro de prompts estruturados

Os prompts abaixo são um roteiro reproduzível para orientar a construção, a revisão e a validação desta solução.

### Prompt 1 - Contexto e construção da base
```text
Atue como um parceiro de programação em Python para um estudante de Ciência
da Computação. Preciso desenvolver uma calculadora simples de terminal para 
uma atividade sobre colaboração entre humanos e IA.

Comece com uma versão básica usando um loop while e as quatro operações.
Explique o funcionamento e aponte as limitações dessa primeira versão 
(como a falta de tratamento de erros), sem tratá-la como produto pronto.
```

### Prompt 2 - Refatoração com critérios verificáveis
```text
Revise a versão inicial. Separe cada operação matemática em uma função 
própria. Adicione tratamento específico para não quebrar a aplicação caso 
ocorra uma divisão por zero ou caso o usuário digite texto (ValueError).

Adicione também operações de exponenciação e raiz quadrada. Mantenha o 
código legível e explique quais alterações são de arquitetura (modularização) 
e quais são de segurança (try/except).
```

### Prompt 3 - Validação da solução
```text
Crie testes com unittest para as funções matemáticas isoladas. 
Cubra soma, divisão segura (garantindo que lança ValueError ao invés de 
ZeroDivisionError), e teste a raiz quadrada de número negativo.
Execute os testes sem depender de input do usuário.
```


## 5. Solução final e decisões de implementação

O código completo da versão final está dividido nas próximas células por responsabilidade.

| Parte | Responsabilidade | Motivo da escolha |
|---|---|---|
| `Funções Matemáticas` | Realizar cálculos isolados | Permite testar a lógica matemática sem passar pela interface do terminal. |
| `Tratamento de Exceções` | Interceptar zeros e negativos | A função matemática lança um erro controlado (ValueError), não abortando o app. |
| `Loop Principal` | Interagir com o usuário e capturar erros de digitação | O bloco try/except intercepta textos digitados sem quebrar o loop `while`. |

### Comparação entre as versões

| Critério | Versão inicial | Versão final |
|---|---|---|
| Arquitetura | Tudo no bloco while | Funções matemáticas isoladas |
| Divisão por zero | `ZeroDivisionError` (Crash) | Tratado, exibe aviso e continua |
| Digitação de texto | `ValueError` (Crash) | Capturado pelo `try/except` geral, pede novo número |
| Funcionalidades | 4 operações | 6 operações (inclui raiz e exp) |
| Testabilidade | Difícil (depende de input) | Fácil (funções recebem parâmetros puros) |


### 5.1. Funções Matemáticas Básicas


In [ ]:
def adicao(a: float, b: float) -> float:
    return a + b

def subtracao(a: float, b: float) -> float:
    return a - b

def multiplicacao(a: float, b: float) -> float:
    return a * b


### 5.2. Funções com Tratamento Específico
A IA sugeriu o uso de `math` para precisão e ajudou a criar barreiras lógicas para divisões impossíveis e raízes complexas.


In [ ]:
def divisao_segura(a: float, b: float) -> float:
    if b == 0:
        raise ValueError("Divisão por zero não é permitida.")
    return a / b

def raiz_quadrada(a: float) -> float:
    if a < 0:
        raise ValueError("Raiz de número negativo não suportada nos reais.")
    return math.sqrt(a)

def exponenciacao(base: float, exp: float) -> float:
    return math.pow(base, exp)


### 5.3. Loop Principal (Interface)
Onde o humano orquestra a máquina. Aqui capturamos os erros que as funções matemáticas (ou o interpretador, no caso de float inválido) lançam.


In [ ]:
def calcular_final():
    print("\n=== Calculadora Inteligente ===")
    print("Opções: +, -, *, /, exp, raiz | '0' para sair")
    
    # Simulação controlada de inputs para o notebook rodar até o fim se chamado
    # Em produção, usa-se while True. Para evitar loop infinito nos testes:
    pass 

# A implementação real do loop, encapsulada para não travar o notebook
def executar_loop_interativo(mock_inputs=None):
    entradas = iter(mock_inputs) if mock_inputs else None
    
    def input_seguro(prompt):
        if entradas:
            val = next(entradas)
            print(prompt + val)
            return val
        return input(prompt)

    while True:
        try:
            op = input_seguro("\nSua operação: ").strip().lower()
        except StopIteration:
            break
            
        if op == '0':
            print("Encerrando...")
            break
            
        if op in ['+', '-', '*', '/', 'exp']:
            try:
                num1 = float(input_seguro("Primeiro número: "))
                num2 = float(input_seguro("Segundo número: "))
            except ValueError:
                print("Entrada inválida! Digite apenas números.")
                continue
                
            try:
                if op == '+': print(f"Resultado: {adicao(num1, num2)}")
                elif op == '-': print(f"Resultado: {subtracao(num1, num2)}")
                elif op == '*': print(f"Resultado: {multiplicacao(num1, num2)}")
                elif op == '/': print(f"Resultado: {divisao_segura(num1, num2)}")
                elif op == 'exp': print(f"Resultado: {exponenciacao(num1, num2)}")
            except ValueError as e:
                print(f"Erro de operação: {e}")
                
        elif op == 'raiz':
            try:
                num = float(input_seguro("Número: "))
                print(f"Resultado: {raiz_quadrada(num)}")
            except ValueError as e:
                print(f"Erro de operação: {e}")
        else:
            print("Operação não reconhecida.")


### 5.4. Exemplos rápidos das funções


In [ ]:
print("Teste Soma:", adicao(10, 5))
print("Teste Divisão Segura:", divisao_segura(10, 2))

try:
    divisao_segura(10, 0)
except ValueError as erro:
    print("Capturou corretamente:", erro)


### 5.5. Comparação com a versão final
Usando a mesma entrada que quebrou a base (`/`, `10`, `0`), a versão refatorada exibe a mensagem de erro elegante e continua o loop (aqui, simulamos também a opção `0` em seguida para sair).


In [ ]:
saida_io = io.StringIO()
with redirect_stdout(saida_io):
    executar_loop_interativo(['/', '10', '0', '0'])

saida_final = saida_io.getvalue()
print(saida_final)
assert "Erro de operação: Divisão por zero não é permitida." in saida_final
assert "ZeroDivisionError" not in saida_final


## 6. Validação Automática com Unittest

As classes abaixo executam testes determinísticos nas funções matemáticas para assegurar a integridade das regras.


In [ ]:
class TestCalculadora(unittest.TestCase):
    def test_adicao(self):
        self.assertEqual(adicao(2.5, 2.5), 5.0)

    def test_divisao_segura(self):
        self.assertEqual(divisao_segura(10, 2), 5.0)
        with self.assertRaisesRegex(ValueError, "Divisão por zero não é permitida"):
            divisao_segura(10, 0)
            
    def test_raiz_quadrada(self):
        self.assertEqual(raiz_quadrada(9), 3.0)
        with self.assertRaisesRegex(ValueError, "Raiz de número negativo"):
            raiz_quadrada(-4)
            
    def test_exponenciacao(self):
        self.assertEqual(exponenciacao(2, 3), 8.0)


In [ ]:
suite_calc = unittest.defaultTestLoader.loadTestsFromTestCase(TestCalculadora)
res = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_calc)
print("\nValidação concluída:", "Aprovada" if res.wasSuccessful() else "Falhou")


## 7. Uso interativo opcional

Para usar a calculadora interativamente, mude `CALCULADORA_INTERATIVA` para `True` e rode a célula. O padrão é `False` para o notebook rodar direto do começo ao fim.


In [ ]:
CALCULADORA_INTERATIVA = False

if CALCULADORA_INTERATIVA:
    executar_loop_interativo()
else:
    print("Para interagir, altere a variável para True e rode a célula.")


## 8. Análise crítica dos exemplos de refatoração do material

### Soma de lista

Trocar um `while` com índice por `sum(numeros)` deixa a intenção mais clara e elimina o controle manual do índice. Para uma lista de inteiros, ambas as versões percorrem os elementos e têm complexidade de tempo **O(n)**. Portanto, reduzir o número de linhas não significa mudar automaticamente a ordem de complexidade.


In [ ]:
numeros = [10, 20, -5]
soma = sum(numeros)
assert soma == 25
assert sum([]) == 0

print("Exemplo executado: verificações aprovadas.")


### Verificação de número primo

O material mostra a redução da busca por divisores até a raiz quadrada do número. A ideia funciona porque, se um número composto é escrito como produto de dois fatores, pelo menos um deles é menor ou igual à sua raiz quadrada.

Porém, as versões apresentadas precisam de uma verificação para `n < 2`. Sem ela, `0` e `1` acabam classificados como primos; na versão com `sqrt`, números negativos ainda podem provocar erro. A solução abaixo usa `isqrt`, que calcula a raiz quadrada inteira sem conversão para ponto flutuante.


In [ ]:
from math import isqrt

def eh_primo(n: int) -> bool:
    if n < 2:
        return False
    for divisor in range(2, isqrt(n) + 1):
        if n % divisor == 0:
            return False
    return True

assert not eh_primo(-7)
assert not eh_primo(0)
assert not eh_primo(1)
assert eh_primo(2)
assert eh_primo(97)
assert not eh_primo(49)

print("Exemplo executado: verificações aprovadas.")


A função recebe números inteiros. A contagem de candidatos a divisor cai de **O(n)** para **O(√n)** no pior caso, considerando cada operação aritmética com custo constante. O `+ 1` é necessário para incluir a raiz inteira no teste e identificar quadrados perfeitos, como `49`.

### Inversão de string

O fatiamento `texto[::-1]` expressa a inversão de forma curta e legível. A versão que coloca cada letra no início de uma nova string copia trechos progressivamente maiores, tendo custo quadrático; o fatiamento percorre o texto com custo linear e produz uma nova string.


In [ ]:
def inverte_texto(texto: str) -> str:
    return texto[::-1]

assert inverte_texto("python") == "nohtyp"
assert inverte_texto("") == ""
assert inverte_texto("a") == "a"

print("Exemplo executado: verificações aprovadas.")


Essa inversão opera sobre os pontos de código da string. Ela atende aos exemplos simples da aula, mas não resolve sozinha a inversão visual de todos os emojis compostos ou caracteres com marcas combinantes.

Esses exemplos mostram por que a revisão continua necessária: um código pode ficar menor ou mais eficiente e ainda ter um erro nas condições de entrada.


## 9. Reflexão sobre a colaboração humano-IA

### Em que a IA contribuiu?
A IA foi crucial para acelerar a construção do *boilerplate* matemático e sugerir implementações seguras que impedissem os travamentos mais comuns (uso adequado do `ValueError` e integração segura com a biblioteca `math`). A criação de testes automatizados unitários também foi facilitada e estruturada via prompt.

### O que precisa continuar sob decisão humana?
A arquitetura do loop, a definição da experiência do usuário e as mensagens de interface precisam de supervisão total. A IA inicialmente tentou colocar blocos de `try/except` generalistas, mas como humano, instruí a separação cirúrgica entre falha de digitação de interface e erro matemático específico.

### Qual foi a melhoria mais relevante?
A separação das funções matemáticas fora do loop principal. No código base original, testar a soma requeria iniciar o prompt de digitação interativo. Com a abstração orientada no prompt, posso validar a matemática isoladamente em milissegundos via Unittest.

### O que levo da atividade?
Que a IA é um copiloto incansável para sintaxe e correção veloz, mas se o humano não assumir as rédeas da arquitetura (`Tech Lead`), o código pode ficar funcional, porém mal acoplado. A estruturação prévia via prompts claros previne essas anomalias.


## 10. Conclusão da entrega

O notebook reúne a base vulnerável e a versão refatorada e segura da Calculadora. Demonstramos ativamente os travamentos na versão antiga, mitigando-os com refatorações sugeridas pela IA. 

O arquivo roda do início ao fim sem interrupções por possuir entradas simuladas, mas está preparado para uso manual caso solicitado na respectiva célula. As responsabilidades da aplicação foram divididas conforme as premissas de boas práticas.


## 11. Material de referência

VALÉRIA, Kadidja. **Programação Assistida por Inteligência Artificial: o novo paradigma da colaboração humano-máquina.** Material da disciplina, UDF, disponibilizado em [Collaborative_AI_Programming_AtividadePrática.pdf](Collaborative_AI_Programming_AtividadePrática.pdf).

Foram utilizados o ciclo de colaboração, os exemplos de refatoração, o desafio de miniprojeto de nível básico e o roteiro de prompts recomendados pela documentação.
